# Asset Survival Analysis Model
### Dataset: `survival_snapshots_train.csv` & `survival_snapshots_test.csv`

---

| Property | Value |
|---|---|
| Dataset | survival_snapshots_*.csv |
| Model | Weibull Accelerated Failure Time (AFT) |
| Input features | Component health, vibration, downtime, temperatures, etc. |
| Target | Time to next maintenance (duration_days) & event indicator |
| Accuracy Metric | **Concordance Index (C-Index)** |
| Expected Accuracy | **0.80 - 0.95+** |


## Step 1 — Install Dependencies

In [ ]:
!pip install -q pandas numpy lifelines

## Step 2 — Imports and Configuration

In [ ]:
import json
import pickle
from datetime import datetime
import pandas as pd
from lifelines import WeibullAFTFitter
from lifelines.utils import concordance_index
import common as C

PENALIZER = 0.02


## Step 3 — Load Dataset

In [ ]:
train_raw = pd.read_csv(C.TRAIN_CSV)
test_raw  = pd.read_csv(C.TEST_CSV)

print(f"Training rows: {len(train_raw):,}")
print(f"Testing rows:  {len(test_raw):,}")


## Step 4 — Define Training Function

In [ ]:
def train_one(train_raw: pd.DataFrame, test_raw: pd.DataFrame, component: str, penalizer: float):
    schema = C.build_schema(train_raw, component)

    # Design matrices (identical transform for train + test).
    X_train = C.transform_design(train_raw[C.feature_cols_for_component(component)], schema)
    X_test  = C.transform_design(test_raw[C.feature_cols_for_component(component)], schema)

    dur_tr, evt_tr = C.derive_target(train_raw, component)
    dur_te, evt_te = C.derive_target(test_raw, component)

    fit_df = X_train.copy()
    fit_df[C.DURATION_COL] = dur_tr.values
    fit_df[C.EVENT_COL]    = evt_tr.values

    aft = WeibullAFTFitter(penalizer=penalizer)
    aft.fit(fit_df, duration_col=C.DURATION_COL, event_col=C.EVENT_COL)

    # Concordance on held-out vehicles: higher predicted median => later failure.
    pred_median = aft.predict_median(X_test)
    c_index = concordance_index(
        event_times=dur_te.values,
        predicted_scores=pred_median.values,
        event_observed=evt_te.values,
    )

    # Top covariates on lambda_ (scale) — drivers of median life.
    try:
        lam = aft.params_.xs("lambda_", level=0).drop(labels=["Intercept"], errors="ignore")
        ranked = lam.abs().sort_values(ascending=False).head(6).index
        top = [{"feature": f, "coef": round(float(lam[f]), 4)} for f in ranked]
    except Exception:
        top = []

    metrics = {
        "component":        component,
        "n_train_rows":     int(len(X_train)),
        "n_test_rows":      int(len(X_test)),
        "n_train_events":   int(evt_tr.sum()),
        "n_test_events":    int(evt_te.sum()),
        "event_rate_train": round(float(evt_tr.mean()), 4),
        "n_features":       len(schema["feature_cols"]),
        "log_likelihood":   round(float(aft.log_likelihood_), 2),
        "AIC":              round(float(aft.AIC_), 2),
        "weibull_rho":      round(float(aft.rho_.iloc[0]) if hasattr(aft, "rho_") else float("nan"), 4),
        "test_concordance": round(float(c_index), 4),
        "top_covariates":   top,
    }
    return aft, schema, metrics


## Step 5 — Train All 5 Components & Evaluate

In [ ]:
C.MODELS_DIR.mkdir(parents=True, exist_ok=True)
bundle = {"version": "survival-v3.0", "components": {}, "component_list": C.COMPONENTS}
all_metrics = []

for comp in C.COMPONENTS:
    print(f"\n── Training {comp} " + "─" * 44)
    aft, schema, metrics = train_one(train_raw, test_raw, comp, PENALIZER)

    with open(C.MODELS_DIR / f"{comp}_aft.pkl", "wb") as f:
        pickle.dump(aft, f)
    with open(C.MODELS_DIR / f"{comp}_features.json", "w") as f:
        json.dump(schema, f, indent=2)

    bundle["components"][comp] = {"model": aft, "schema": schema}

    print(f"  rows train/test : {metrics['n_train_rows']:,} / {metrics['n_test_rows']:,}")
    print(f"  events tr/te    : {metrics['n_train_events']:,} / {metrics['n_test_events']:,} (rate {metrics['event_rate_train']:.1%})")
    print(f"  AIC             : {metrics['AIC']}")
    print(f"  test c-index    : {metrics['test_concordance']}")

    all_metrics.append(metrics)


## Step 6 — Save Unified Bundle & Report
This mimics how the production script dumps the models for the FastAPI backend.

In [ ]:
with open(C.MODELS_DIR / "survival_bundle.pkl", "wb") as f:
    pickle.dump(bundle, f)

report = {
    "trained_at": datetime.utcnow().isoformat() + "Z",
    "version":    "survival-v3.0",
    "dataset":    C.DATASET_PATH.name,
    "penalizer":  PENALIZER,
    "components": all_metrics,
}
with open(C.MODELS_DIR / "training_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\n── Final Accuracy Summary " + "─" * 40)
print(f"{'component':<12s} {'c-index':>9s} {'events tr/te':>16s} {'AIC':>12s}")
for m in all_metrics:
    ev = f"{m['n_train_events']}/{m['n_test_events']}"
    print(f"  {m['component']:<10s} {m['test_concordance']:>9.4f} {ev:>16s} {m['AIC']:>12.0f}")
